# Perth Crime Hotspot Analysis

**Data source:** WA Police Force — *Crime Statistics by Suburb* ([data.wa.gov.au](https://data.wa.gov.au))

This notebook walks through the full analysis pipeline:
1. Data loading and preprocessing
2. Exploratory data analysis (EDA)
3. Crime clustering by type and frequency (K-Means)
4. Geospatial heatmaps and visualisations

**Colab-friendly workflow:** the notebook resolves the public CSV resource at runtime and loads it directly with Pandas, so you do not need to upload a file manually on mobile. If you want a temporary copy in the current Colab session, set `SAVE_TO_COLAB_RUNTIME = True` in the next section.

---

In [ ]:
import json
import os
from pathlib import Path
from urllib.request import urlopen
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import folium
from folium.plugins import HeatMap, MarkerCluster
import plotly.express as px
import plotly.graph_objects as go

plt.rcParams["figure.dpi"] = 120
print("Libraries loaded successfully.")

## 1. Load Raw Data

This section looks up the dataset metadata on the WA Open Data portal, extracts the public CSV resource URL, and reads it directly with `pandas.read_csv(...)`. This avoids the usual Colab file-upload step and is easier to run from a mobile device.

If you want a temporary copy inside the current Colab runtime, set `SAVE_TO_COLAB_RUNTIME = True` before running the cell below.

In [ ]:
DATASET_API_URL = os.environ.get(
    "WA_CRIME_DATASET_API_URL",
    "https://data.wa.gov.au/api/3/action/package_show?id=crime-statistics-by-suburb",
)
SAVE_TO_COLAB_RUNTIME = globals().get("SAVE_TO_COLAB_RUNTIME", False)
COLAB_RUNTIME_CSV_PATH = os.environ.get(
    "WA_CRIME_RUNTIME_CSV_PATH",
    "/content/perth_crime_statistics.csv",
)



def resolve_public_csv_url(dataset_api_url: str = DATASET_API_URL) -> str:
    with urlopen(dataset_api_url, timeout=30) as response:
        payload = json.load(response)

    if not payload.get("success"):
        raise ValueError("WA Open Data API did not return a successful response.")

    for resource in payload.get("result", {}).get("resources", []):
        resource_url = resource.get("url")
        resource_format = str(resource.get("format", "")).lower()
        if resource_url and "csv" in resource_format:
            return resource_url

    raise ValueError("Could not find a public CSV resource for the WA crime dataset.")


csv_url = resolve_public_csv_url()
print(f"Resolved CSV URL: {csv_url}")
df_raw = pd.read_csv(csv_url)
print(f"Shape: {df_raw.shape}")

if SAVE_TO_COLAB_RUNTIME:
    Path(COLAB_RUNTIME_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)
    df_raw.to_csv(COLAB_RUNTIME_CSV_PATH, index=False)
    print(f"Saved a runtime copy to: {COLAB_RUNTIME_CSV_PATH}")

df_raw.head()

## 2. Preprocess

In [ ]:
# Standardise column names
df = df_raw.copy()
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[\s/]+", "_", regex=True)
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

# Enforce types
df["offence_count"] = pd.to_numeric(df["offence_count"], errors="coerce")
df["latitude"]      = pd.to_numeric(df["latitude"],      errors="coerce")
df["longitude"]     = pd.to_numeric(df["longitude"],     errors="coerce")
df = df.dropna(subset=["suburb", "offence_subdivision", "offence_count", "latitude", "longitude"])
df = df[df["offence_count"] > 0]
df["period"] = df["period"].str.upper()

print(f"Cleaned rows: {len(df):,}")
df.dtypes

## 3. Exploratory Data Analysis

In [ ]:
# Total offences by suburb
suburb_totals = df.groupby("suburb")["offence_count"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
suburb_totals.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Total Offences by Suburb (2022/23)")
ax.set_xlabel("Suburb")
ax.set_ylabel("Total Offences")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Offences by type
type_totals = df.groupby("offence_subdivision")["offence_count"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
type_totals.plot(kind="bar", ax=ax, color="darkorange", edgecolor="white")
ax.set_title("Total Offences by Type (2022/23)")
ax.set_xlabel("Offence Subdivision")
ax.set_ylabel("Total Offences")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 4. Build Suburb Feature Matrix

In [ ]:
pivot = (
    df.groupby(["suburb", "district", "local_government_area",
                "latitude", "longitude", "offence_division"])["offence_count"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)
pivot.columns.name = None
offence_cols = [c for c in pivot.columns
                if c not in ["suburb", "district", "local_government_area",
                              "latitude", "longitude"]]
pivot["total_offences"] = pivot[offence_cols].sum(axis=1)
pivot.head()

## 5. K-Means Clustering

In [ ]:
X = pivot[offence_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow + Silhouette
ks = range(2, 9)
inertias, sil_scores = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_ = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(ks), inertias, marker="o", color="steelblue")
axes[0].set_title("Elbow Curve"); axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
axes[1].plot(list(ks), sil_scores, marker="s", color="darkorange")
axes[1].set_title("Silhouette Scores"); axes[1].set_xlabel("k"); axes[1].set_ylabel("Score")
plt.tight_layout(); plt.show()

best_k = list(ks)[int(np.argmax(sil_scores))]
print(f"Best k = {best_k}")

In [ ]:
K = 4  # chosen number of clusters
km = KMeans(n_clusters=K, random_state=42, n_init=10)
pivot["cluster"] = km.fit_predict(X_scaled)

# Assign risk tier names by mean total offences
cluster_means = pivot.groupby("cluster")["total_offences"].mean().sort_values(ascending=False)
tiers = ["High Crime", "Moderate-High Crime", "Moderate Crime", "Low Crime"]
tier_map = {cid: tiers[i] for i, cid in enumerate(cluster_means.index)}
pivot["risk_tier"] = pivot["cluster"].map(tier_map)

print("Cluster sizes:")
print(pivot.groupby(["cluster", "risk_tier"])["total_offences"].agg(["count", "mean"]))

In [ ]:
# PCA scatter
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
colors_map = {"High Crime": "#d73027", "Moderate-High Crime": "#fc8d59",
              "Moderate Crime": "#fee08b", "Low Crime": "#91cf60"}
for tier, color in colors_map.items():
    mask = pivot["risk_tier"] == tier
    ax.scatter(coords[mask, 0], coords[mask, 1], color=color, label=tier, s=80, alpha=0.85)
    for _, row in pivot[mask].iterrows():
        idx = pivot.index.get_loc(row.name)
        ax.annotate(row["suburb"], (coords[idx, 0], coords[idx, 1]),
                    fontsize=7, ha="center", va="bottom")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title(f"K-Means Clusters (k={K}) — PCA Projection")
ax.legend(title="Risk Tier"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Geospatial Heatmap (Folium)

In [ ]:
PERTH_LAT, PERTH_LON = -31.9505, 115.8605

fmap = folium.Map(location=[PERTH_LAT, PERTH_LON], zoom_start=11,
                  tiles="CartoDB positron")
heat_data = [[row["latitude"], row["longitude"], row["total_offences"]]
             for _, row in pivot.iterrows()]
HeatMap(heat_data, min_opacity=0.4, radius=30, blur=20,
        gradient={0.2: "blue", 0.5: "lime", 0.8: "yellow", 1.0: "red"}).add_to(fmap)
fmap

## 7. Cluster Marker Map (Folium)

In [ ]:
TIER_COLORS = {"High Crime": "#d73027", "Moderate-High Crime": "#fc8d59",
               "Moderate Crime": "#fee08b", "Low Crime": "#91cf60"}

fmap2 = folium.Map(location=[PERTH_LAT, PERTH_LON], zoom_start=11,
                   tiles="CartoDB positron")
mc = MarkerCluster().add_to(fmap2)

for _, row in pivot.iterrows():
    tier  = row["risk_tier"]
    color = TIER_COLORS.get(tier, "#999")
    popup = (f"<b>{row['suburb']}</b><br>District: {row['district']}<br>"
             f"LGA: {row['local_government_area']}<br>"
             f"Risk Tier: <span style='color:{color};font-weight:bold;'>{tier}</span><br>"
             f"Total Offences: {int(row['total_offences'])}")
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=max(5, min(20, row["total_offences"] / 50)),
        color=color, fill=True, fill_color=color, fill_opacity=0.7,
        popup=folium.Popup(popup, max_width=220),
        tooltip=f"{row['suburb']} — {tier}",
    ).add_to(mc)

fmap2

## 8. Interactive Bar Chart (Plotly)

In [ ]:
fig = px.bar(
    pivot.nlargest(15, "total_offences").sort_values("total_offences"),
    x="total_offences", y="suburb", orientation="h",
    color="risk_tier", color_discrete_map=TIER_COLORS,
    title="Top 15 Perth Suburbs by Total Offences (2022/23)",
    labels={"total_offences": "Total Offences", "suburb": "Suburb", "risk_tier": "Risk Tier"},
    template="plotly_white",
)
fig.show()

## 9. Summary

| Risk Tier | Description |
|-----------|-------------|
| **High Crime** | Very high total offences — concentrated urban/commercial activity |
| **Moderate-High Crime** | Significant crime load — typically inner/middle suburbs |
| **Moderate Crime** | Average levels — outer suburbs with moderate density |
| **Low Crime** | Below-average offences — typically newer or lower-density areas |

---
*Data sourced from WA Police Force crime statistics via the public CSV resource published at [data.wa.gov.au](https://data.wa.gov.au). If the WA Open Data portal is unavailable, the repository also includes a smaller sample CSV for offline experimentation.*